# Temporal validation: can the models predict NEW bridges?

The headline comparison uses a random 80/20 split, so every construction era appears on
both sides. This notebook answers the stricter deployment question: **train only on
bridges built before a cutoff year, then predict bridges built after it** — i.e., score
structures whose era the model has never seen.

**Design** (cutoff 2015, robustness check at 2010):
- **temporal train** = the standard seed-42 80% training split MINUS every bridge built
  ≥ 2015. Keeping the standard membership means the only change vs the shipped models is
  the removal of post-cutoff bridges.
- **new-bridge test** = ALL bridges built ≥ 2015 (32,769 bridges, 96 events, max
  observed age 10) — none appear in training, whichever half of the standard split they
  came from.
- **in-era control** = the standard 20% test split restricted to pre-2015 bridges — the
  same model scored in its own era, for reference.
- Hyperparameters are identical to the shipped runs (`train_national_rsf`,
  `train_national_parametric`); the parametric impute/standardize path is mirrored from Notebook 6.

**Read the numbers with three caveats:**
1. Only **96 events** exist among post-2015 bridges (they are young), so the new-bridge
   C-index carries a percentile-bootstrap 95% CI (B=1000), and a 2010 cutoff
   (303 events) is reported as a robustness line for Cox/AFT.
2. The observable horizon for the new cohort is ≤ 10 years — this validates
   early-life risk ranking, not long-horizon prediction.
3. Trees cannot extrapolate `YEAR_BUILT_027` past the training range (every post-cutoff
   bridge lands in the "newest" leaves); Cox/AFT extrapolate the linear term. Both are
   real properties of deploying each model on new construction, not artifacts.

Outputs -> `us_temporal_validation.json` (stage-saved, restart-safe).


In [1]:
# ── Configuration + shipped-run references ───────────────────────────────────
import json
import time
import gc
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored
from lifelines import CoxPHFitter, WeibullAFTFitter

DATA_CLEAN = Path("us_rsf_data_clean.csv")          # shipped RSF matrix
DATA_PARAM = Path("us_parametric_data_clean.csv")   # shipped parametric matrix
RSF_METRICS  = Path("us_rsf_metrics.json")
PARA_METRICS = Path("us_parametric_metrics.json")
OUT_JSON   = Path("us_temporal_validation.json")

KEYS = ["STATE_FIPS", "STRUCTURE_NUMBER_008"]
SMOKE_TEST = False   # True -> _smoke inputs/outputs + tiny forest
CUTOFF = 2015        # professor-specified boundary; 2010 robustness line below
BOOT_B = 1000

RSF_PARAMS = dict(n_estimators=200, max_samples=0.10, min_samples_split=100,
                  min_samples_leaf=50, max_features="sqrt", low_memory=True,
                  n_jobs=-1, random_state=42)   # = train_national_rsf full-scale config
PENALIZER = 0.01                                # train_national_parametric.ipynb
AFT_MAXITER = 2000

if SMOKE_TEST:
    if Path("us_rsf_data_clean_smoke.csv").exists():
        DATA_CLEAN   = Path("us_rsf_data_clean_smoke.csv")
        DATA_PARAM   = Path("us_parametric_data_clean_smoke.csv")
        RSF_METRICS  = Path("us_rsf_metrics_smoke.json")
        PARA_METRICS = Path("us_parametric_metrics_smoke.json")
    OUT_JSON = OUT_JSON.with_stem(OUT_JSON.stem + "_smoke")
    RSF_PARAMS.update(n_estimators=25, max_samples=0.5)
    BOOT_B = 200
    print(f"SMOKE_TEST: {DATA_CLEAN} -> {OUT_JSON}")

def save_stage(key, value):
    """Persist each heavy stage to OUT_JSON immediately (resume-safe after a restart)."""
    data = json.loads(OUT_JSON.read_text()) if OUT_JSON.exists() else {}
    data[key] = value
    data["generated_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    OUT_JSON.write_text(json.dumps(data, indent=2))

rsf_ref  = json.loads(RSF_METRICS.read_text())
para_ref = json.loads(PARA_METRICS.read_text())
N_TEST_REF = rsf_ref["n_test"]
assert rsf_ref["n_test"] == para_ref["n_test"], "shipped runs disagree on the test split"

def c_or_none(event, time_, risk):
    """C-index, or None when a cohort has no comparable pairs (tiny smoke subsets)."""
    try:
        return float(concordance_index_censored(event, time_, risk)[0])
    except Exception as e:
        print(f"  C-index not estimable: {type(e).__name__}")
        return None

def boot_ci(event, time_, risk, seed=2026):
    """Percentile bootstrap over test rows (risk scores fixed — no refits)."""
    rng = np.random.default_rng(seed)
    n, vals = len(risk), []
    for _ in range(BOOT_B):
        idx = rng.integers(0, n, n)
        try:
            vals.append(concordance_index_censored(event[idx], time_[idx], risk[idx])[0])
        except Exception:
            continue
    if len(vals) < BOOT_B * 0.5:      # too many degenerate resamples to trust a CI
        return None
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return [round(float(lo), 4), round(float(hi), 4)]

save_stage("design", {
    "cutoff": CUTOFF,
    "split": ("standard seed-42 80/20 membership; temporal train = 80% split minus "
              "bridges built >= cutoff; new-bridge test = ALL bridges built >= cutoff; "
              "in-era control = 20% split restricted to pre-cutoff bridges"),
    "hyperparameters": "identical to the shipped full-scale runs",
    "bootstrap_B": BOOT_B,
})
print(f"cutoff {CUTOFF}; shipped n_test reference {N_TEST_REF:,}")


cutoff 2015; shipped n_test reference 194,781


In [2]:
# ── Temporal split builder ───────────────────────────────────────────────────
def temporal_sets(path, cutoff):
    """X/y for temporal train, in-era control, and new-bridge test at a given cutoff."""
    cols = pd.read_csv(path, nrows=0).columns
    dtypes = {c: "float32" for c in cols}
    dtypes.update({k: str for k in KEYS})
    dtypes["event"] = "int8"
    m = pd.read_csv(path, dtype=dtypes)
    yb = pd.to_numeric(m["YEAR_BUILT_027"], errors="coerce").to_numpy()
    assert not np.isnan(yb).any(), "YEAR_BUILT_027 must be complete in the clean matrix"

    y = Surv.from_arrays(event=m["event"].astype(bool).to_numpy(),
                         time=m["time"].astype("float64").to_numpy())
    X = m.drop(columns=["event", "time", "bridge_age"] + KEYS)   # same X as the shipped runs
    del m
    gc.collect()

    # Same n + same seed -> identical split membership to the shipped models.
    idx_train, idx_test = train_test_split(np.arange(len(X)), test_size=0.2, random_state=42)
    assert len(idx_test) == N_TEST_REF, \
        "split parity with the shipped runs failed — matrices out of sync, rebuild first"
    tr_mask = np.zeros(len(X), dtype=bool)
    tr_mask[idx_train] = True

    old, new = yb < cutoff, yb >= cutoff
    sets = {"train": (X[tr_mask & old], y[tr_mask & old]),
            "ctrl":  (X[~tr_mask & old], y[~tr_mask & old]),
            "new":   (X[new], y[new])}
    for name, (Xs, ys) in sets.items():
        print(f"  {name:<6} n {len(Xs):>9,}   events {int(ys['event'].sum()):>7,}")
    return sets

print("temporal_sets defined")


temporal_sets defined


In [3]:
# ── RSF at shipped full-scale hyperparameters (~1.5 h at national scale) ─────
sets = temporal_sets(DATA_CLEAN, CUTOFF)
X_tr, y_tr = sets["train"]

t0 = time.time()
rsf = RandomSurvivalForest(**RSF_PARAMS)
rsf.fit(X_tr, y_tr)
fit_min = (time.time() - t0) / 60

res = {"cutoff": CUTOFF, "n_train": int(len(X_tr)),
       "train_events": int(y_tr["event"].sum()), "fit_minutes": round(fit_min, 1)}
for name, key in [("ctrl", "in_era_control"), ("new", "new_bridges")]:
    Xs, ys = sets[name]
    risk = np.concatenate([rsf.predict(Xs.iloc[i:i + 50_000])
                           for i in range(0, len(Xs), 50_000)])
    c = c_or_none(ys["event"], ys["time"], risk)
    entry = {"n": int(len(Xs)), "events": int(ys["event"].sum()),
             "c_index": round(c, 4) if c is not None else None}
    if name == "new":
        entry["c_index_ci95"] = boot_ci(ys["event"], ys["time"], risk)
    res[key] = entry
    print(f"RSF {key}: C {entry['c_index']}   (n={entry['n']:,}, events={entry['events']:,})")
save_stage("rsf", res)

del rsf, sets, X_tr, y_tr
gc.collect()


  train  n   752,897   events 227,941
  ctrl   n   188,239   events  57,045
  new    n    32,769   events      96


RSF in_era_control: C 0.7536   (n=188,239, events=57,045)


RSF new_bridges: C 0.8365   (n=32,769, events=96)


105

In [4]:
# ── Cox + Weibull AFT (same code path as train_national_parametric) ──────────
def run_parametric_temporal(cutoff, save_prefix):
    sets = temporal_sets(DATA_PARAM, cutoff)
    X_tr, y_tr = sets["train"]
    X_ct, y_ct = sets["ctrl"]
    X_nw, y_nw = sets["new"]

    # Missingness flags with the Notebook-6 dedup rule — computed on the temporal TRAIN
    # set (the model's world), then applied to every set as a per-row transform.
    na_cols = X_tr.columns[X_tr.isna().any()].tolist()
    never_reconstructed = (X_tr["ever_reconstructed"] == 0).to_numpy().astype("int8")
    seen, flag_cols = set(), []
    for c in na_cols:
        msk = X_tr[c].isna().to_numpy()
        key = msk.tobytes()
        if key in seen:
            continue
        seen.add(key)
        f = msk.astype("int8")
        if (f == never_reconstructed).all():
            continue
        flag_cols.append(c)

    def add_flags(Z):
        return pd.concat([Z, pd.DataFrame({c + "_missing": Z[c].isna().astype("int8")
                                           for c in flag_cols}, index=Z.index)], axis=1)
    X_tr, X_ct, X_nw = add_flags(X_tr), add_flags(X_ct), add_flags(X_nw)

    # Train-median imputation over ALL columns: test cohorts can have NaN in columns the
    # temporal train observed completely (the reverse of the Notebook-6 situation).
    med = X_tr.median()
    X_tr, X_ct, X_nw = X_tr.fillna(med), X_ct.fillna(med), X_nw.fillna(med)
    const = X_tr.columns[(X_tr.max(axis=0) == X_tr.min(axis=0)).to_numpy()].tolist()
    if const:
        X_tr, X_ct, X_nw = (Z.drop(columns=const) for Z in (X_tr, X_ct, X_nw))

    def score(name, risk_fn):
        out = {}
        for key, Xs, ys in [("in_era_control", X_ct, y_ct), ("new_bridges", X_nw, y_nw)]:
            risk = risk_fn(Xs)
            c = c_or_none(ys["event"], ys["time"], risk)
            entry = {"n": int(len(Xs)), "events": int(ys["event"].sum()),
                     "c_index": round(c, 4) if c is not None else None}
            if key == "new_bridges":
                entry["c_index_ci95"] = boot_ci(ys["event"], ys["time"], risk)
            out[key] = entry
            print(f"{name} {key}: C {entry['c_index']}   "
                  f"(n={entry['n']:,}, events={entry['events']:,})")
        return out

    base = {"cutoff": cutoff, "n_train": int(len(X_tr)),
            "train_events": int(y_tr["event"].sum())}

    # Cox
    train_df = X_tr.assign(time=y_tr["time"], event=y_tr["event"].astype("int8"))
    t0 = time.time()
    cph = CoxPHFitter(penalizer=PENALIZER)
    cph.fit(train_df, duration_col="time", event_col="event")
    print(f"Cox fit {(time.time()-t0):.0f}s")
    cox_res = dict(base)
    cox_res.update(score("Cox", lambda Z: cph.predict_partial_hazard(Z).to_numpy()))
    save_stage(save_prefix + "_cox", cox_res)

    # Weibull AFT (train-stat standardization of continuous features, as Notebook 6)
    is_binary = ((X_tr == 0) | (X_tr == 1)).all()
    cont = X_tr.columns[~is_binary]
    mu, sd = X_tr[cont].mean(), X_tr[cont].std()

    def std(Z):
        Z = Z.copy()
        Z[cont] = (Z[cont] - mu) / sd
        return Z

    train_df_aft = std(X_tr).assign(time=y_tr["time"], event=y_tr["event"].astype("int8"))
    t0 = time.time()
    aft = WeibullAFTFitter(penalizer=PENALIZER)
    try:
        aft.fit(train_df_aft, duration_col="time", event_col="event",
                fit_options={"maxiter": AFT_MAXITER})
    except Exception as e:
        print(f"AFT retry at 10x penalizer after {type(e).__name__}")
        aft = WeibullAFTFitter(penalizer=10 * PENALIZER)
        aft.fit(train_df_aft, duration_col="time", event_col="event",
                fit_options={"maxiter": AFT_MAXITER})
    print(f"AFT fit {(time.time()-t0)/60:.1f} min")
    aft_res = dict(base)
    aft_res.update(score("AFT", lambda Z: -aft.predict_median(std(Z)).to_numpy(dtype="float64")))
    save_stage(save_prefix + "_aft", aft_res)

    del sets, X_tr, X_ct, X_nw, train_df, train_df_aft
    gc.collect()

run_parametric_temporal(CUTOFF, "temporal")


  train  n   752,897   events 227,941
  ctrl   n   188,239   events  57,045
  new    n    32,769   events      96


C:\Users\Joshu\AppData\Local\Temp\ipykernel_32660\494816817.py:55: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df = X_tr.assign(time=y_tr["time"], event=y_tr["event"].astype("int8"))


C:\Users\Joshu\Documents\GitHub\Bridge_project_pt2\.venv\Lib\site-packages\lifelines\utils\__init__.py:1100: ConvergenceWarning: Column(s) ['STRUCTURE_TYPE_043B_Other', 'TOLL_020_Other'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.

  warnings.warn(dedent(warning_text), ConvergenceWarning)


Cox fit 50s


Cox in_era_control: C 0.8835   (n=188,239, events=57,045)


Cox new_bridges: C 0.7931   (n=32,769, events=96)


AFT fit 4.5 min


AFT in_era_control: C 0.8834   (n=188,239, events=57,045)


AFT new_bridges: C 0.7972   (n=32,769, events=96)


In [5]:
# ── Robustness: 2010 cutoff — parametric models only ─────────────────────────
# The 2015 new-bridge cohort has just 96 events; 2010 (303 events) checks that the
# conclusion is not an artifact of the thin cohort. RSF omitted: a coarse-forest refit
# costs ~1.5 h and the parametric models are the headline at national scale.
if not SMOKE_TEST:
    run_parametric_temporal(2010, "robustness_2010")
else:
    print("SMOKE_TEST: robustness cutoff skipped")


  train  n   735,771   events 227,779
  ctrl   n   183,852   events  57,000
  new    n    54,282   events     303


C:\Users\Joshu\AppData\Local\Temp\ipykernel_32660\494816817.py:55: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df = X_tr.assign(time=y_tr["time"], event=y_tr["event"].astype("int8"))


C:\Users\Joshu\Documents\GitHub\Bridge_project_pt2\.venv\Lib\site-packages\lifelines\utils\__init__.py:1100: ConvergenceWarning: Column(s) ['STRUCTURE_TYPE_043B_Other', 'TOLL_020_Other'] have very low variance. This may harm convergence. 1) Are you using formula's? Did you mean to add '-1' to the end. 2) Try dropping this redundant column before fitting if convergence fails.

  warnings.warn(dedent(warning_text), ConvergenceWarning)


Cox fit 50s


Cox in_era_control: C 0.8838   (n=183,852, events=57,000)


Cox new_bridges: C 0.7804   (n=54,282, events=303)


AFT fit 4.4 min


AFT in_era_control: C 0.8833   (n=183,852, events=57,000)


AFT new_bridges: C 0.788   (n=54,282, events=303)


In [6]:
# ── Summary ──────────────────────────────────────────────────────────────────
data = json.loads(OUT_JSON.read_text())
rows = [("RSF", data.get("rsf")), ("Cox PH", data.get("temporal_cox")),
        ("Weibull AFT", data.get("temporal_aft"))]
print(f"cutoff {CUTOFF} — trained on pre-cutoff bridges only\n")
print(f"{'model':<14} {'in-era C':>9} {'new-bridge C':>13} {'95% CI':>19}")
for name, d in rows:
    if not d:
        continue
    ci = d["new_bridges"].get("c_index_ci95")
    ci_s = f"[{ci[0]:.4f}, {ci[1]:.4f}]" if ci else "-"
    print(f"{name:<14} {str(d['in_era_control']['c_index']):>9} "
          f"{str(d['new_bridges']['c_index']):>13} {ci_s:>19}")

rb_cox, rb_aft = data.get("robustness_2010_cox"), data.get("robustness_2010_aft")
if rb_cox and rb_aft:
    print(f"\n2010 cutoff (robustness, {rb_cox['new_bridges']['events']} events): "
          f"Cox new-bridge C {rb_cox['new_bridges']['c_index']}   "
          f"AFT {rb_aft['new_bridges']['c_index']}")
print(f"\nsaved {OUT_JSON}")


cutoff 2015 — trained on pre-cutoff bridges only

model           in-era C  new-bridge C              95% CI
RSF               0.7536        0.8365    [0.7859, 0.8797]
Cox PH            0.8835        0.7931    [0.7305, 0.8431]
Weibull AFT       0.8834        0.7972    [0.7386, 0.8425]

2010 cutoff (robustness, 303 events): Cox new-bridge C 0.7804   AFT 0.788

saved us_temporal_validation.json
